In [2]:
from dotenv import load_dotenv
import os 
import pygsheets
import requests
import pandas as pd
from pangres import upsert
from sqlalchemy import text, create_engine



load_dotenv()
###Spreedsheet loading###
service_account = pygsheets.authorize(service_account_file="JSONs\\turningpointlol-485813-8d21cf2ec3aa.json")

sheet = service_account.open_by_url("https://docs.google.com/spreadsheets/d/1MyhKZyeGs6qXk9ZY8nSniDfrMSIXt9KWVlGjD-vaGR0/edit?usp=sharing")
###Spreadsheetloading###

api_key = os.environ.get("riot_api_key") #loads api key from .env file

###Leaderboard for highest ranks###
root = "https://euw1.api.riotgames.com/"
chall = "lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5"
grandmaster = "lol/league/v4/grandmasterleagues/by-queue/RANKED_SOLO_5x5"
master = "lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5"


chall_response = requests.get(root + chall + "?api_key=" + api_key)
gm_response = requests.get(root + grandmaster + "?api_key=" + api_key)
master_response = requests.get(root + master + "?api_key=" + api_key)


chall_df = pd.DataFrame(chall_response.json()["entries"])
grandmaster_df = pd.DataFrame(gm_response.json()["entries"])
master_df = pd.DataFrame(master_response.json()["entries"])
###Leaderboard for highest ranks###

#puuid is the unique identifier tag for individual players
def get_puuid(summonerId = None, gameName=None, tagline=None, region='europe'):

    if summonerId is not None:
        root_url = f"https://{region}.api.riotgames.com/"
        endpoint = "lol/summoner/v4/summoners"
        print(root_url+endpoint+"?api_key="+api_key)
        response =  requests.get(root_url+endpoint+"?api_key="+api_key)

        return response.json()["puuid"]
    else:
        root_url = f"https://{region}.api.riotgames.com/"
        endpoint = f"riot/account/v1/accounts/by-riot-id/{gameName}/{tagline}"

        response = requests.get(root_url+endpoint+"?api_key="+api_key)
        print(response)#shows response from server, check riot dev docs for response list
        return response.json()['puuid']
    
def get_id_from_puuid(puuid=None, region="euw1"):
    root_url = f"https://{region}.api.riotgames.com/"
    endpoint = "riot/account/v1/accounts/by-puuid/"

    response = requests.get(root_url+endpoint+"?api_key"+api_key)

    id = {
        "gameName" : response.json()["gameName"],
        "tagline" : response.json()["tagLine"]
    }

    return id

def get_match_history(region=None, puuid=None, start=0, count=20):

    root_url = f"https://{region}.api.riotgames.com/"
    endpoint = f"lol/match/v5/matches/by-puuid/{puuid}/ids"
    query_params = f"?start={start}&count={count}"
    response = requests.get(root_url+endpoint+query_params+"&api_key="+api_key)
    print(response)
    return response.json()

def get_match_data_from_id(region=None, matchId=None):
    root_url = f"https://{region}.api.riotgames.com/"
    endpoint = f"lol/match/v5/matches/{matchId}"

    response = requests.get(root_url+endpoint+"?api_key="+api_key)

    return response.json()

def process_match_json(game, puuid):


    side_dict = {
        100:'blue',
        200:'red'
    }
    try:
        metadata = game['metadata']
        matchId = metadata['matchId']
        participants = metadata['participants']

        info = game['info']
        player = info['participants'][participants.index(puuid)]

        gameCreation = info['gameCreation']
        gameStartTimestamp = info['gameStartTimestamp']
        gameEndTimestamp = info['gameEndTimestamp']
        timePlayed = (gameEndTimestamp - gameStartTimestamp) / 1000
        gameMode = info['gameMode']
        gameVersion = info['gameVersion']
        platformId = info['platformId']
        queueId = info['queueId']
        puuid = player['puuid']
        riotIdGameName = player['summonerName']
        try:
            riotIdTagLine = player['riotIdTagline']
        except:
            riotIdTagLine = ''
        side = side_dict[player['teamId']]
        win = player['win']

        champion = player['championName']
        kills = player['kills']
        deaths = player['deaths']
        assists = player['assists']
        summOne = player['summoner1Id']
        summTwo = player['summoner2Id']
        earlySurrender = player['gameEndedInEarlySurrender']
        surrender = player['gameEndedInSurrender']
        firstBlood = player['firstBloodKill']
        firstBloodAssist = player['firstBloodAssist']
        firstTower = player['firstTowerKill']
        firstTowerAssist = player['firstTowerAssist']
        dragonKills = player['dragonKills']

        damageDealtToBuildings = player['damageDealtToBuildings']
        damageDealtToObjectives = player['damageDealtToObjectives']
        damageSelfMitigated = player['damageSelfMitigated']
        goldEarned = player['goldEarned']
        teamPosition = player['teamPosition']
        lane = player['lane']
        largestKillingSpree = player['largestKillingSpree']
        longestTimeSpentLiving = player['longestTimeSpentLiving']
        objectivesStolen = player['objectivesStolen']
        totalMinionsKilled = player['totalMinionsKilled']
        totalAllyJungleMinionsKilled = player['totalAllyJungleMinionsKilled']
        totalEnemyJungleMinionsKilled = player['totalEnemyJungleMinionsKilled']
        totalNeutralMinionsKilled = totalAllyJungleMinionsKilled + totalEnemyJungleMinionsKilled
        totalDamageDealtToChampions = player['totalDamageDealtToChampions']
        totalDamageShieldedOnTeammates = player['totalDamageShieldedOnTeammates']
        totalHealsOnTeammates = player['totalHealsOnTeammates']
        totalDamageTaken = player['totalDamageTaken']
        totalTimeCCDealt = player['totalTimeCCDealt']
        totalTimeSpentDead = player['totalTimeSpentDead']
        turretKills = player['turretKills']
        turretsLost = player['turretsLost']
        visionScore = player['visionScore']
        controlWardsPlaced = player['detectorWardsPlaced']
        wardsKilled = player['wardsKilled']
        wardsPlaced = player['wardsPlaced']

        item0 = player['item0']
        item1 = player['item1']
        item2 = player['item2']
        item3 = player['item3']
        item4 = player['item4']
        item5 = player['item5']
        item6 = player['item6']

        perks = player['perks']

        perkKeystone = perks['styles'][0]['selections'][0]['perk']
        perkPrimaryRow1 = perks['styles'][0]['selections'][1]['perk']
        perkPrimaryRow2 = perks['styles'][0]['selections'][2]['perk']
        perkPrimaryRow3 = perks['styles'][0]['selections'][3]['perk']
        perkPrimaryStyle = perks['styles'][0]['style']
        perkSecondaryRow1 = perks['styles'][1]['selections'][0]['perk']
        perkSecondaryRow2 = perks['styles'][1]['selections'][1]['perk']
        perkSecondaryStyle = perks['styles'][1]['style']
        perkShardDefense = perks['statPerks']['defense']
        perkShardFlex = perks['statPerks']['flex']
        perkShardOffense = perks['statPerks']['offense']

        matchDF = pd.DataFrame({
            'match_id': [matchId],
            'participants': [participants],
            'game_creation': [gameCreation],
            'game_start_timestamp': [gameStartTimestamp],
            'game_end_timestamp': [gameEndTimestamp],
            'game_version': [gameVersion],
            'queue_id': [queueId],
            'game_mode': [gameMode],
            'platform_id': [platformId],
            'puuid': [puuid],
            'riot_id': [riotIdGameName],
            'riot_tag': [riotIdTagLine],
            'time_played': [timePlayed],
            'side': [side],
            'win': [win],
            'team_position': [teamPosition],
            'lane': [lane],
            'champion': [champion],
            'kills': [kills],
            'deaths': [deaths],
            'assists': [assists],
            'summoner1_id': [summOne],
            'summoner2_id': [summTwo],
            'gold_earned': [goldEarned],
            'total_minions_killed': [totalMinionsKilled],
            'total_neutral_minions_killed': [totalNeutralMinionsKilled],
            'total_ally_jungle_minions_killed': [totalAllyJungleMinionsKilled],
            'total_enemy_jungle_minions_killed': [totalEnemyJungleMinionsKilled],
            'early_surrender': [earlySurrender],
            'surrender': [surrender],
            'first_blood': [firstBlood],
            'first_blood_assist': [firstBloodAssist],
            'first_tower': [firstTower],
            'first_tower_assist': [firstTowerAssist],
            'damage_dealt_to_buildings': [damageDealtToBuildings],
            'turret_kills': [turretKills],
            'turrets_lost': [turretsLost],
            'damage_dealt_to_objectives': [damageDealtToObjectives],
            'dragonKills': [dragonKills],
            'objectives_stolen': [objectivesStolen],
            'longest_time_spent_living': [longestTimeSpentLiving],
            'largest_killing_spree': [largestKillingSpree],
            'total_damage_dealt_champions': [totalDamageDealtToChampions],
            'total_damage_taken': [totalDamageTaken],
            'total_damage_self_mitigated': [damageSelfMitigated],
            'total_damage_shielded_teammates': [totalDamageShieldedOnTeammates],
            'total_heals_teammates': [totalHealsOnTeammates],
            'total_time_crowd_controlled': [totalTimeCCDealt],
            'total_time_spent_dead': [totalTimeSpentDead],
            'vision_score': [visionScore],
            'wards_killed': [wardsKilled],
            'wards_placed': [wardsPlaced],
            'control_wards_placed': [controlWardsPlaced],
            'item0': [item0],
            'item1': [item1],
            'item2': [item2],
            'item3': [item3],
            'item4': [item4],
            'item5': [item5],
            'item6': [item6],
            'perk_keystone': [perkKeystone],
            'perk_primary_row_1': [perkPrimaryRow1],
            'perk_primary_row_2': [perkPrimaryRow2],
            'perk_primary_row_3': [perkPrimaryRow3],
            'perk_secondary_row_1': [perkSecondaryRow1],
            'perk_secondary_row_2': [perkSecondaryRow2],
            'perk_primary_style': [perkPrimaryStyle],
            'perk_secondary_style': [perkSecondaryStyle],
            'perk_shard_defense': [perkShardDefense],
            'perk_shard_flex': [perkShardFlex],
            'perk_shard_offense': [perkShardOffense],
        })

        return matchDF
    except:
        return pd.DataFrame()



match_ids= get_match_history(region ="europe", puuid= "mA3AR323uxw3kTmgNL0q0qDjZFihqtzXwxcKI2DiJIw9Lwg3gLrZjNTfE2kCcTZphLg-GfcXJgyPcg")



df = pd.DataFrame()
for match_id in match_ids:
    game = get_match_data_from_id(region="europe", matchId=match_id)
    matchDF = process_match_json(game, puuid="mA3AR323uxw3kTmgNL0q0qDjZFihqtzXwxcKI2DiJIw9Lwg3gLrZjNTfE2kCcTZphLg-GfcXJgyPcg")

    df = pd.concat([df, matchDF])





###community dragon links all here###
perk = "https://raw.communitydragon.org/latest/plugins/rcp-be-lol-game-data/global/default/v1/perks.json"
perk_style = "https://raw.communitydragon.org/latest/plugins/rcp-be-lol-game-data/global/default/v1/perkstyles.json"
items = "https://raw.communitydragon.org/latest/plugins/rcp-be-lol-game-data/global/default/v1/items.json"
###community dragon links all here###

perk_json = requests.get(perk).json()
perk_styles_json = requests.get(perk_style).json()
items_json = requests.get(items).json()

#function takes from community dragon, and uses dictionary to change API given ID's into their in game name, eg Keystone id: 80401 turns into Grasp of the undying
def json_extract(obj, key):
    
    arr = []

    def extract(obj, arr, key):
        if isinstance(obj, dict):
             for k, v in obj.items():
                 if k == key:
                     arr.append(v)
                 elif isinstance(v, (dict, list)):
                    extract(v, arr, key)
        elif isinstance (obj, list):
            for item in obj:
                extract(item, arr, key)

        return arr
    
    values = extract(obj, arr, key)
    return values

perk_ids = json_extract(perk_json, "id")
perk_names = json_extract(perk_json, "name")
perk_styles_ids = json_extract(perk_styles_json, "id")
perk_styles_names = json_extract(perk_styles_json, "name")
items_ids = json_extract(items_json, "id")
items_name = json_extract(items_json, "name")

perk_dict = dict(map(lambda i, j : (int(i), j), perk_ids, perk_names))
perk_styles_dict = dict(map(lambda i, j : (int(i), j), perk_styles_ids, perk_styles_names))
item_dict = dict(map(lambda i, j : (int(i), j), items_ids, items_name))

pd.options.display.max_columns = 100
combined_perk_dict = {**perk_dict, **perk_styles_dict}

#where perk in dataframe replace
perk_columns = [
    'perk_keystone', 'perk_primary_row_1', 'perk_primary_row_2', 'perk_primary_row_3',
    'perk_secondary_row_1', 'perk_secondary_row_2', 'perk_primary_style', 'perk_secondary_style',
    'perk_shard_defense', 'perk_shard_flex', 'perk_shard_offense'
]
items_columns = [
    "item0", "item1", "item2", "item3", "item4", "item5", "item6"
]
def find_column(selectColumns=None, selectDict=None):
    for col in selectColumns:
        if col in df.columns:
            df[col] = df[col].replace(selectDict)










db_username=os.environ.get("db_username")
db_password=os.environ.get("db_password")
db_host=os.environ.get("db_host")
db_port=os.environ.get("db_port")
db_name=os.environ.get("db_name")


def create_db_connection_string(db_username, db_password, db_host, db_port, db_name):
    connection_url = "postgresql+psycopg2://"+db_username+":"+db_password+"@"+db_host+":"+db_port+"/"+db_name
    return connection_url

conn_url = create_db_connection_string(db_username, db_password, db_host, db_port, db_name)

db_engine = create_engine(conn_url, pool_recycle=3600)

connection = db_engine.connect()

<Response [200]>


In [ ]:
upsert(con=connection, df=df, schema="soloq", table_name="regional_player_matches", create_table=True, create_schema=True, if_row_exists="update")

In [4]:
find_column(items_columns, item_dict)
find_column(perk_columns, combined_perk_dict)

In [5]:
df

,match_id,participants,game_creation,game_start_timestamp,game_end_timestamp,game_version,queue_id,game_mode,platform_id,puuid,riot_id,riot_tag,time_played,side,win,team_position,lane,champion,kills,deaths,assists,summoner1_id,summoner2_id,gold_earned,total_minions_killed,total_neutral_minions_killed,total_ally_jungle_minions_killed,total_enemy_jungle_minions_killed,early_surrender,surrender,first_blood,first_blood_assist,first_tower,first_tower_assist,damage_dealt_to_buildings,turret_kills,turrets_lost,damage_dealt_to_objectives,dragonKills,objectives_stolen,longest_time_spent_living,largest_killing_spree,total_damage_dealt_champions,total_damage_taken,total_damage_self_mitigated,total_damage_shielded_teammates,total_heals_teammates,total_time_crowd_controlled,total_time_spent_dead,vision_score,wards_killed,wards_placed,control_wards_placed,item0,item1,item2,item3,item4,item5,item6,perk_keystone,perk_primary_row_1,perk_primary_row_2,perk_primary_row_3,perk_secondary_row_1,perk_secondary_row_2,perk_primary_style,perk_secondary_style,perk_shard_defense,perk_shard_flex,perk_shard_offense
0,EUW1_7714347614,[c8xUjK9iBCuq1NMgXJRPnEfi3chl97kCemVNSDkYnJTmp...,1770121289421,1770121308535,1770122236434,16.2.741.3171,420,CLASSIC,EUW1,mA3AR323uxw3kTmgNL0q0qDjZFihqtzXwxcKI2DiJIw9Lw...,,21591,927.899,red,False,TOP,NONE,Varus,0,6,0,4,21,4264,96,0,0,0,False,True,False,False,False,False,588,0,2,2661,0,0,245,0,5948,10376,4467,0,0,70,140,4,0,2,0,Doran's Blade,Blasting Wand,Dark Seal,Kindlegem,Mercury's Treads,Dagger,Stealth Ward,Press the Attack,Triumph,Legend: Alacrity,Last Stand,Bone Plating,Overgrowth,Domination,Resolve,Health Scaling,Adaptive Force,Attack Speed
0,EUW1_7714315849,[mA3AR323uxw3kTmgNL0q0qDjZFihqtzXwxcKI2DiJIw9L...,1770119186639,1770119215962,1770120788518,16.2.741.3171,420,CLASSIC,EUW1,mA3AR323uxw3kTmgNL0q0qDjZFihqtzXwxcKI2DiJIw9Lw...,,21591,1572.556,blue,False,TOP,JUNGLE,Renekton,2,6,0,4,14,9800,214,0,0,0,False,False,False,False,False,False,6643,1,9,6643,0,0,371,0,29437,36244,25689,0,0,16,200,17,1,6,0,Doran's Blade,Mercury's Treads,Long Sword,Black Cleaver,Executioner's Calling,Sundered Sky,Stealth Ward,Press the Attack,Triumph,Legend: Alacrity,Last Stand,Second Wind,Overgrowth,Domination,Resolve,Health Scaling,Adaptive Force,Adaptive Force
0,EUW1_7714286679,[Sed_5t3j1iGFBEDvvUSwtXFINVNlEE_Mxts-I-tgfowKk...,1770116867452,1770116961457,1770118965320,16.2.741.3171,420,CLASSIC,EUW1,mA3AR323uxw3kTmgNL0q0qDjZFihqtzXwxcKI2DiJIw9Lw...,,21591,2003.863,red,True,TOP,JUNGLE,Varus,13,4,5,4,21,16621,251,4,0,4,False,False,False,False,False,False,15210,4,2,20096,0,0,509,6,35737,25334,22653,0,0,255,179,28,0,14,0,Shattered Armguard,Dusk and Dawn,Plated Steelcaps,Nashor's Tooth,Mejai's Soulstealer,Riftmaker,Stealth Ward,Press the Attack,Triumph,Legend: Alacrity,Last Stand,Bone Plating,Overgrowth,Domination,Resolve,Health Scaling,Adaptive Force,Attack Speed
0,EUW1_7712618823,[tms4uXBAoWEC7WrAavQaqUpkJ4MVX8ScONME0Umn7Ocmi...,1769995585034,1769995694712,1769997324917,16.2.741.3171,1700,CHERRY,EUW1,mA3AR323uxw3kTmgNL0q0qDjZFihqtzXwxcKI2DiJIw9Lw...,,21591,1630.205,blue,True,,BOTTOM,Rakan,8,5,13,2202,2201,15920,0,0,0,0,False,False,False,False,False,False,0,0,0,0,0,0,104,8,52706,56568,72283,6091,5608,315,849,0,0,0,0,Unending Despair,Ionian Boots of Lucidity,Pyromancer's Cloak,Sunfire Aegis,Liandry's Anguish,Runecarver,Arcane Sweeper,0,0,0,0,0,0,0,0,0,0,0
0,EUW1_7709326957,[AjLABXxL8GDIyla-ZcL4ZvUfUcoJLfFHIXZ9NB0S9I1CD...,1769815580289,1769815615364,1769817312411,16.2.741.3171,1700,CHERRY,EUW1,mA3AR323uxw3kTmgNL0q0qDjZFihqtzXwxcKI2DiJIw9Lw...,,21591,1697.047,red,False,,MIDDLE,Anivia,8,8,8,2202,2201,18125,0,0,0,0,False,False,False,False,False,False,0,0,0,0,0,0,111,6,67609,76158,49193,0,0,1067,1158,0,0,0,0,Everfrost,Sorcerer's Shoes,Rabadon's Deathcap,Liandry's Anguish,Void Staff,Detonation Orb,Arcane Sweeper,0,0,0,0,0,0,0,0,0,0,0
0,EUW1_7709286864,[88Q0bY7PClBd-5Vxg_gjp2E2zFr6etpDIpiOrBZc2IJUJ...,1769814133145,1769814201064,1769815860327,16.2.741.3171,

In [ ]:
get_puuid(gameName="MrHawtrerGames", tagline="21591")

<Response [200]>


'mA3AR323uxw3kTmgNL0q0qDjZFihqtzXwxcKI2DiJIw9Lwg3gLrZjNTfE2kCcTZphLg-GfcXJgyPcg'

In [12]:
game = get_match_data_from_id(region="europe", matchId="EUW1_7709326957")

In [14]:
get_match_history(region ="europe", puuid="mA3AR323uxw3kTmgNL0q0qDjZFihqtzXwxcKI2DiJIw9Lwg3gLrZjNTfE2kCcTZphLg-GfcXJgyPcg", start=0, count=20)

<Response [200]>


['EUW1_7714347614',
 'EUW1_7714315849',
 'EUW1_7714286679',
 'EUW1_7712618823',
 'EUW1_7709326957',
 'EUW1_7709286864',
 'EUW1_7708263999',
 'EUW1_7708235668',
 'EUW1_7708203734',
 'EUW1_7704870049',
 'EUW1_7704821861',
 'EUW1_7704766525',
 'EUW1_7704098019',
 'EUW1_7697876615',
 'EUW1_7697835557',
 'EUW1_7697218040',
 'EUW1_7697150057',
 'EUW1_7697070233',
 'EUW1_7696437691',
 'EUW1_7696425323']

In [16]:
pd.concat([chall_df, grandmaster_df, master_df])

,puuid,leaguePoints,rank,wins,losses,veteran,inactive,freshBlood,hotStreak
0,qk_b-7WO5vaeAwH7d7F9dkqKhelgDEPOTAiii39a6Dl0tJ...,2102,I,181,133,True,False,False,True
1,elVqsEc8Msl_K4mN_3FwXf8Y9Us6n_Hw2foqy120tluCLm...,2091,I,122,75,True,False,False,True
2,93EHgfAA0ZguymGQrlwTyZnBycYnk11Acxqin7TPb8TCne...,2051,I,150,90,True,False,False,True
3,s_KGdcr0iAbgmtr7b2zndo1i9-PhMbbOMJ5Mou1YT_7jjB...,2027,I,121,57,True,False,False,False
4,pyOzMEpaJldIxELcsqVtmTR-BaXqwMGYi1Uh80hi0-F_M1...,2023,I,187,98,True,False,False,False
...,...,...,...,...,...,...,...,...,...
9995,UhXnu-KprMsywC7KJUQl6YVGVjzTN-Cdw9JPNBImzbykA7...,116,I,50,58,False,False,True,True
9996,A8H90z99ShIPvJXUvpoJ8aSD1N0Nft-c_QqVZzzuQmogcS...,116,I,41,36,False,False,False,False
9997,VU7TiHABZEQLR4lIRfoSO142G1IbU-b2kF6Hjk9KvCkEA6...,116,I,40,32,False,False,True,False
9998,2DIjC_2LbDVQ3UJgP1Auxi33Y9NAuNGYusn1PvbBMyqcnh...,116,I,40,48,False,False,False,False


In [50]:
process_match_json(game, "mA3AR323uxw3kTmgNL0q0qDjZFihqtzXwxcKI2DiJIw9Lwg3gLrZjNTfE2kCcTZphLg-GfcXJgyPcg")

,match_id,participants,game_creation,game_start_timestamp,game_end_timestamp,game_version,queue_id,game_mode,platform_id,puuid,...,perk_primary_row_1,perk_primary_row_2,perk_primary_row_3,perk_secondary_row_1,perk_secondary_row_2,perk_primary_style,perk_secondary_style,perk_shard_defense,perk_shard_flex,perk_shard_offense
0,EUW1_7696425323,[mA3AR323uxw3kTmgNL0q0qDjZFihqtzXwxcKI2DiJIw9L...,1769091980007,1769092042747,1769092122623,16.2.738.9684,420,CLASSIC,EUW1,mA3AR323uxw3kTmgNL0q0qDjZFihqtzXwxcKI2DiJIw9Lw...,...,9111,9104,8299,8451,8444,8000,8400,5001,5008,5008


In [37]:
pd.options.display.max_columns = 100
combined_dict = {**perk_dict, **perk_styles_dict}

# Check what we're working with
print("Before replacement:")
print(df[['perk_keystone', 'perk_primary_style']].head())

# WORKAROUND: Replace column by column instead of whole DataFrame
perk_columns = [
    'perk_keystone', 'perk_primary_row_1', 'perk_primary_row_2', 'perk_primary_row_3',
    'perk_secondary_row_1', 'perk_secondary_row_2', 'perk_primary_style', 'perk_secondary_style',
    'perk_shard_defense', 'perk_shard_flex', 'perk_shard_offense'
]

for col in perk_columns:
    if col in df.columns:
        df[col] = df[col].replace(combined_dict)

print("\nAfter replacement:")
print(df[['perk_keystone', 'perk_primary_style']].head())

Before replacement:
   perk_keystone  perk_primary_style
0           8005                8000

After replacement:
      perk_keystone perk_primary_style
0  Press the Attack         Domination
